# Phase 3: Portfolio Optimization

## Project context

This notebook uses the saved daily return data from Phase 1 and the asset universe from Phase 2 to build simple long-only portfolio allocations. It does not run full backtesting or create a dashboard.

## Optimization objective

The goal is to compare three interpretable portfolio construction methods:

- Equal-weight portfolio
- Minimum-volatility portfolio
- Maximum-Sharpe portfolio

SPY is excluded from optimized asset portfolios and used only as the benchmark reference from prior phases.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.config import OUTPUTS_DIR, FIGURES_DIR
from src.optimization import (
    LAST_OPTIMIZATION_STATUS,
    load_returns,
    select_asset_returns,
    annualize_expected_returns,
    annualize_covariance,
    equal_weight_portfolio,
    minimum_volatility_portfolio,
    maximum_sharpe_portfolio,
    generate_random_portfolios,
    summarize_portfolio,
)
from src.visualization import (
    plot_portfolio_weights,
    plot_efficient_frontier_simulation,
    plot_portfolio_risk_return,
    plot_allocation_pie_or_bar,
)

BENCHMARK = "SPY"
RETURNS_PATH = OUTPUTS_DIR / "returns" / "daily_returns.csv"
PORTFOLIOS_DIR = OUTPUTS_DIR / "portfolios"
PORTFOLIOS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def _normalize(values):
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    if not finite.any():
        return np.zeros_like(values)
    vmin = np.nanmin(values[finite])
    vmax = np.nanmax(values[finite])
    if np.isclose(vmin, vmax):
        return np.full_like(values, 0.5, dtype=float)
    return (values - vmin) / (vmax - vmin)


def save_plotly_or_pillow(fig, output_path, chart_type, data):
    output_path = Path(output_path)
    try:
        fig.write_image(str(output_path), width=1200, height=700, scale=2)
        return "plotly"
    except Exception as exc:
        draw_basic_png(output_path, chart_type, data, str(exc))
        return "pillow_fallback"


def draw_basic_png(output_path, chart_type, data, reason):
    width, height = 1200, 700
    margin = 80
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    draw.text((margin, 24), output_path.stem.replace("_", " ").title(), fill="black", font=font)
    draw.text((margin, 48), "Rendered with Pillow fallback because Plotly PNG export was unavailable.", fill="#555555", font=font)
    left, top, right, bottom = margin, 110, width - margin, height - margin
    draw.rectangle((left, top, right, bottom), outline="#333333")

    if chart_type == "bar":
        frame = data.copy()
        values = frame["value"].astype(float).values
        labels = frame["label"].astype(str).tolist()
        scale_values = _normalize(values)
        bar_width = (right - left) / max(len(values), 1) * 0.65
        for i, value in enumerate(values):
            x = left + (right - left) * (i + 0.5) / len(values)
            y = bottom - (bottom - top) * scale_values[i]
            draw.rectangle((x - bar_width / 2, y, x + bar_width / 2, bottom), fill="#1f77b4")
            draw.text((x - 18, bottom + 10), labels[i][:8], fill="black", font=font)
            draw.text((x - 18, y - 16), f"{value:.2f}", fill="black", font=font)
    elif chart_type == "scatter":
        frame = data.copy()
        x = _normalize(frame["annualized_volatility"].values)
        y = _normalize(frame["annualized_return"].values)
        labels = frame.get("portfolio", pd.Series([""] * len(frame))).astype(str).tolist()
        for i in range(len(frame)):
            px = left + (right - left) * x[i]
            py = bottom - (bottom - top) * y[i]
            draw.ellipse((px - 4, py - 4, px + 4, py + 4), fill="#1f77b4")
            if labels[i]:
                draw.text((px + 7, py - 7), labels[i][:18], fill="black", font=font)
    image.save(output_path)


## Load daily returns

In [ ]:
returns = load_returns(RETURNS_PATH)
asset_returns = select_asset_returns(returns, benchmark=BENCHMARK)
display(asset_returns.head())
print(f"Asset return shape: {asset_returns.shape}")
print(f"Date range: {asset_returns.index.min().date()} to {asset_returns.index.max().date()}")

## Asset universe review

In [ ]:
asset_names = asset_returns.columns.tolist()
print(asset_names)
print(f"Benchmark excluded from optimization: {BENCHMARK not in asset_names}")

## Expected returns and covariance

In [ ]:
expected_returns = annualize_expected_returns(asset_returns)
covariance_matrix = annualize_covariance(asset_returns)
display(expected_returns.to_frame("annualized_expected_return").style.format("{:.2%}"))
display(covariance_matrix.style.format("{:.4f}"))

## Equal-weight portfolio

In [ ]:
equal_weights = equal_weight_portfolio(asset_names)
summarize_portfolio("equal_weight", equal_weights, expected_returns, covariance_matrix)

## Minimum-volatility portfolio

In [ ]:
minimum_volatility_weights = minimum_volatility_portfolio(expected_returns, covariance_matrix)
summarize_portfolio("minimum_volatility", minimum_volatility_weights, expected_returns, covariance_matrix)

## Maximum-Sharpe portfolio

In [ ]:
maximum_sharpe_weights = maximum_sharpe_portfolio(expected_returns, covariance_matrix, risk_free_rate=0.0)
summarize_portfolio("maximum_sharpe", maximum_sharpe_weights, expected_returns, covariance_matrix)

## Random portfolio simulation

In [ ]:
random_portfolios = generate_random_portfolios(
    expected_returns,
    covariance_matrix,
    n_portfolios=5000,
    risk_free_rate=0.0,
    random_state=42,
)
random_portfolios.to_csv(PORTFOLIOS_DIR / "random_portfolios.csv", index=False)
display(random_portfolios.head())
print(f"Random portfolios generated: {len(random_portfolios):,}")

## Efficient frontier-style risk-return analysis

In [ ]:
weights_df = pd.DataFrame({
    "asset": asset_names,
    "equal_weight": equal_weights,
    "minimum_volatility": minimum_volatility_weights,
    "maximum_sharpe": maximum_sharpe_weights,
})
weights_df.to_csv(PORTFOLIOS_DIR / "portfolio_weights.csv", index=False)

portfolio_summary = pd.DataFrame([
    summarize_portfolio("equal_weight", equal_weights, expected_returns, covariance_matrix),
    summarize_portfolio("minimum_volatility", minimum_volatility_weights, expected_returns, covariance_matrix),
    summarize_portfolio("maximum_sharpe", maximum_sharpe_weights, expected_returns, covariance_matrix),
])
portfolio_summary.to_csv(PORTFOLIOS_DIR / "portfolio_summary.csv", index=False)

display(weights_df.style.format({"equal_weight": "{:.2%}", "minimum_volatility": "{:.2%}", "maximum_sharpe": "{:.2%}"}))
display(portfolio_summary.style.format({"annualized_return": "{:.2%}", "annualized_volatility": "{:.2%}", "sharpe_ratio": "{:.2f}"}))
LAST_OPTIMIZATION_STATUS

## Portfolio weight comparison

In [ ]:
max_sharpe_top_asset = weights_df.sort_values("maximum_sharpe", ascending=False).iloc[0]
min_vol_top_asset = weights_df.sort_values("minimum_volatility", ascending=False).iloc[0]
print(f"Largest max-Sharpe allocation: {max_sharpe_top_asset['asset']} ({max_sharpe_top_asset['maximum_sharpe']:.2%})")
print(f"Largest min-vol allocation: {min_vol_top_asset['asset']} ({min_vol_top_asset['minimum_volatility']:.2%})")

## Figures

In [ ]:
figure_specs = {
    "portfolio_weights_comparison.png": (
        plot_portfolio_weights(weights_df),
        "bar",
        weights_df[["asset", "maximum_sharpe"]].rename(columns={"asset": "label", "maximum_sharpe": "value"}),
    ),
    "efficient_frontier_simulation.png": (
        plot_efficient_frontier_simulation(random_portfolios, portfolio_summary),
        "scatter",
        random_portfolios[["annualized_volatility", "annualized_return"]].copy(),
    ),
    "portfolio_risk_return_comparison.png": (
        plot_portfolio_risk_return(portfolio_summary),
        "scatter",
        portfolio_summary.copy(),
    ),
    "max_sharpe_allocation.png": (
        plot_allocation_pie_or_bar(weights_df, "maximum_sharpe"),
        "bar",
        weights_df[["asset", "maximum_sharpe"]].rename(columns={"asset": "label", "maximum_sharpe": "value"}),
    ),
}

figure_export_methods = {}
for filename, (fig, chart_type, data) in figure_specs.items():
    figure_export_methods[filename] = save_plotly_or_pillow(fig, FIGURES_DIR / filename, chart_type, data)

figure_export_methods

## Business interpretation

- Equal weight provides a transparent benchmark allocation across the eight selected assets.
- The minimum-volatility portfolio emphasizes assets that reduced historical portfolio variance under long-only constraints.
- The maximum-Sharpe portfolio emphasizes assets with stronger historical risk-adjusted returns, which can lead to concentration.
- The random portfolio simulation provides an intuitive view of the tradeoff between expected return, volatility, and Sharpe ratio.

## Limitations

- Expected returns and covariance are estimated from historical daily returns and may not persist.
- The risk-free rate is held at zero for simplicity.
- Portfolios are long-only with weights bounded from 0 percent to 100 percent.
- The local SciPy optimizer is unavailable because of a NumPy compatibility issue; this notebook uses a documented deterministic random-search fallback.
- No transaction costs, turnover limits, tax effects, or full out-of-sample backtesting are included in Phase 3.

## Next steps for Phase 4 backtesting

- Convert static allocations into a simple historical portfolio performance comparison.
- Compare portfolio returns, drawdowns, volatility, and benchmark-relative behavior.
- Keep interpretation business-readable and avoid treating historical optimization as a live investment recommendation.